# Logistic Regression Example

Despite its name, logistic regression is a classification algorithm for classifying things as belonging to some category or not-belonging. Because it has no closed-form solution, we train it via gradient descent.

## Logistic Regression Sources

- [Stanford Logistic Regression](https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/pdfs/40%20LogisticRegression.pdf)

## Jax Resources

- [Training Cookbook](https://docs.jax.dev/en/latest/the-training-cookbook.html#achieving-high-performance)
- [Grad Basics](../../exercises/exe_07_grad_basics.ipynb)

In [1]:
import sklearn
from sklearn.datasets import load_iris
import jax
import jax.numpy as jnp
from jax import vmap
from jax import grad, value_and_grad
from jax import random

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

In [2]:
NUM_TRAIN_STEP = 100
learning_rate = 0.001

# Data Setup

- we take two classes of the iris dataset: 0 and 1, to turn it into a binary classification problem

In [3]:
X, y = load_iris(return_X_y=True)
mask = y < 2  # is 0 or 1
X = X[mask]
y = y[mask]
print(f"X-shape: {X.shape}")
print(f"y-shape: {y.shape}")

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)

X-shape: (100, 4)
y-shape: (100,)
(80, 4) (80,)


# Model

## Parameter Setup


In [4]:
key = random.key(42)
W = 0.01 * random.normal(key, (X.shape[1],))
b = 0.01

params = {"W": W, "b": b}

print(f"Initial parameters: {params}")

Initial parameters: {'W': Array([-0.00028305,  0.00467132,  0.00295703,  0.00153546], dtype=float32), 'b': 0.01}


## Math Refresher

We optimize our logistic regression by minimizing the negative log-likelihood function. The log-likelihood here is for our binary classification task, so our probability distribution of choice is the Bernoulli distribution.

$$\hat{y}_i = \sigma(f_{\theta}(x_i))$$

where 

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

and our label probabilities (deriveable from the likelihood) is:

$$log P(D_i | \theta) = y_i log \hat{y}_i + (1 - y_i) log ( 1 - \hat{y}_i)$$

## Boilerplate Training Code

In [8]:
@jax.jit
def sigmoid(z):
    return 1 /  (1 + jnp.e ** -z)
    
@jax.jit
def predict_single(x, params):
    return sigmoid(x @ params["W"] + params["b"])

predict = jax.vmap(predict_single, in_axes=(0, None))
    
@jax.jit
def _log_likelihood(y, X, params):
    y_hat = predict(X, params)
    lhs = y * jnp.log(y_hat)
    rhs = (1 - y) * jnp.log(1 - y_hat)
    return jnp.sum(lhs + rhs)

log_likelihood_and_grad = jax.value_and_grad(_log_likelihood, argnums=2)

def update_params(lr, old_params, grad, mask: dict[str, bool] = None):
    """
    If we "mask" out a value, we prevent the application of the gradient
    """
    new_params = {}
    for k, v in old_params.items():
        if mask is not None and mask.get(k, False):
            new_params[k] = v  # Don't update (masked)
        else:
            new_params[k] = v + lr * grad[k]  # Gradient ascent
    return new_params

def test(i, y, X, params):
    y_hat = predict(X, params)
    matching = jnp.round(y_hat) == y
    return jnp.mean(matching)

In [ ]:
def train(train_data, test_data, initial_params):
    y_train, X_train = train_data
    y_test, X_test = test_data
    params = initial_params
    for i in range(NUM_TRAIN_STEP):
        loss, grad = log_likelihood_and_grad(y, X, params)
        params = update_params(learning_rate, params, grad)
        if i % 10 == 0 and i > 1:
            performance = test(i, y_test, X_test, params)
            print(f"At iteration: {i}, loss was: {loss}, and test-performance was: {performance}")
    return params
            
trained_model_params = train(
    (y_train, X_train), 
    (y_test, X_test), 
    params
)